In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
vipoooool_new_plant_diseases_dataset_path = kagglehub.dataset_download('vipoooool/new-plant-diseases-dataset')

print('Data source import complete.')


# **Import Libraries:**

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Conv2D,Dropout,MaxPooling2D,Flatten
from tensorflow.keras.callbacks import EarlyStopping,ModelCheckpoint
import matplotlib.pyplot as plt
import numpy as np


# **Read Dataset:**

## Read Training Data

In [ ]:
train_data=ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

In [ ]:
train_generator=train_data.flow_from_directory(
    '/kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train/',
    target_size=(150, 150),
    batch_size=128,
    class_mode='categorical',
    subset='training'  # Set to 'training' for training images

)

## Read Validation Data

In [ ]:
validation_generator = train_data.flow_from_directory(
    '/kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train/',
    target_size=(150, 150),
    batch_size=128,
    class_mode='categorical',
    subset='validation'  # Set to 'validation' for validation images
)

## Read Test Data

In [ ]:
test_data=ImageDataGenerator(
    rescale=1./255
)

In [ ]:
test_generator=test_data.flow_from_directory(
    '/kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid/',
    target_size=(150, 150),
    batch_size=128,
    class_mode='categorical'

)

# **Showing Each Class Name and Coressponding Index:**

In [ ]:
# Display the 38 unique class names (labels) each one in seperated line
for label,index in train_generator.class_indices.items():
    print(index," : ",label)


In [ ]:
# Get a batch of images and labels from the train_generator
images, labels = next(train_generator)  # Get the next batch (128 images in this case)

# Reverse the class_indices dictionary to get a mapping from index to class name
class_labels = {v: k for k, v in train_generator.class_indices.items()}

# Convert one-hot encoded labels back to the index form (if using categorical mode)
label_indices = np.argmax(labels, axis=1)

# If you want to shuffle the batch and display 50 random images
random_indices = np.random.choice(len(images), 50, replace=False)  # Choose 50 random indices

# Plot 50 random images with labels
plt.figure(figsize=(20, 20))
for i, idx in enumerate(random_indices):
    plt.subplot(10, 5, i + 1)  # Create a grid of 5 rows and 10 columns
    plt.imshow(images[idx])
    plt.title(class_labels[label_indices[idx]])  # Show the label as the title
    plt.axis('off')  # Hide the axis

plt.show()

# **CNN Model:**

In [ ]:
model=Sequential([
  # Input Layer in CNN
  Conv2D(32,(3,3),activation='relu',input_shape=(150,150,3)),



  # Feture Extraction Layers in CNN with MaxPooling
  Conv2D(32,(3,3),activation='relu'),
  MaxPooling2D((2,2)),
  Conv2D(64,(3,3),activation='relu'),
  MaxPooling2D((2,2)),
  Conv2D(128,(3,3),activation='relu'),
  MaxPooling2D((2,2)),
  Conv2D(256,(3,3),activation='relu'),
  MaxPooling2D((2,2)),
  Conv2D(512,(3,3),activation='relu'),
  MaxPooling2D((2,2)),


# Fully Connected Neural Network
  Flatten(),
  Dense(512, activation='relu'),
  Dropout(0.5),
  Dense(38, activation='softmax')  # 38 classes

])

In [ ]:
model.summary()

# **Plotting CNN Model:**

In [ ]:
from tensorflow.keras.utils import plot_model
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True)

# **Fitting the CNN Model:**

In [ ]:
from tensorflow.keras import metrics
model.compile(optimizer='adam',loss='categorical_crossentropy'
              ,metrics=['accuracy',metrics.Recall(),metrics.Precision()])

In [ ]:
early_stopping=EarlyStopping(monitor='val_loss',patience=5)
model_checkpoint=ModelCheckpoint(
    'best_model.keras',  # File path where the model will be saved
    monitor='val_loss',  # Metric to monitor
    save_best_only=True,  # Save only the model with the best validation loss
    mode='min',  # 'min' because lower loss is better
    verbose=1  # Verbosity mode
)

In [ ]:
history=model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // validation_generator.batch_size,
    epochs=50,
    callbacks=[early_stopping,model_checkpoint],
    verbose=1
)

# **Showing Training and Validation Accuracy of Best Model after Finishing Training Epochs**

In [ ]:
# using max to use best model saved by modelcheckpoint
train_accuracy = max(history.history['accuracy'])
val_accuracy = max(history.history['val_accuracy'])

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

# **Model Evaluation:**

In [ ]:
results = model.evaluate(test_generator)
loss = results[0]
accuracy = results[1]
precision = results[2]
recall = results[3]

print(f'Testing Loss: {loss}')
print(f'Testing Accuracy: {accuracy}')
print(f'Testing Precision: {precision}')
print(f'Testing Recall: {recall}')

# **Visualization of CNN Model Evaluation Metrics:**

In [ ]:

import matplotlib.pyplot as plt

# Create a figure with 1 row and 2 columns for subplots
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

# Plot training and validation accuracy
axs[0].plot(history.history['accuracy'], label='Training Accuracy')
axs[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axs[0].set_xlabel('Epochs')
axs[0].set_ylabel('Accuracy')
axs[0].set_title('Training and Validation Accuracy')
axs[0].legend()

# Plot training and validation loss
axs[1].plot(history.history['loss'], label='Training Loss')
axs[1].plot(history.history['val_loss'], label='Validation Loss')
axs[1].set_xlabel('Epochs')
axs[1].set_ylabel('Loss')
axs[1].set_title('Training and Validation Loss')
axs[1].legend()

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()


In [ ]:
# Get predictions on the validation set
y_probs = model.predict(validation_generator)   # probabilities for each class
y_pred = np.argmax(y_probs, axis=1)             # pick highest probability class


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Step 1: Get predictions
y_probs = model.predict(validation_generator)   # probabilities
y_pred = np.argmax(y_probs, axis=1)             # predicted labels
y_true = validation_generator.classes           # true labels

# Step 2: Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
labels = list(validation_generator.class_indices.keys())

plt.figure(figsize=(12, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# Step 3: Classification Report
print("Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=labels))


# **Evaluation Function For all Train,Valid and Test**

In [ ]:
def Evaluate_model(model, train_generator, validation_generator, test_generator):
    model_evaluate_train = model.evaluate(train_generator)
    print("Loss       : ", model_evaluate_train[0])
    print("Accuracy   : ", model_evaluate_train[1])
    print("Precision  : ", model_evaluate_train[2])
    print("Recall     : ", model_evaluate_train[3])


    model_evaluate_valid = model.evaluate(validation_generator)
    print("Loss       : ", model_evaluate_valid[0])
    print("Accuracy   : ", model_evaluate_valid[1])
    print("Precision  : ", model_evaluate_valid[2])
    print("Recall     : ", model_evaluate_valid[3])


    model_evaluate_test = model.evaluate(test_generator)
    print("Loss       : ", model_evaluate_test[0])
    print("Accuracy   : ", model_evaluate_test[1])
    print("Precision  : ", model_evaluate_test[2])
    print("Recall     : ", model_evaluate_test[3])

    return np.round(model_evaluate_train[0], 2), np.round(model_evaluate_test[0], 2), \
           np.round(model_evaluate_train[1], 2), np.round(model_evaluate_test[1], 2), \
           np.round(model_evaluate_train[2], 2), np.round(model_evaluate_test[2], 2), \
           np.round(model_evaluate_train[3], 2), np.round(model_evaluate_test[3], 2)

# **Print all Evaluation Metrics of both Train,Valid,Testing**

In [ ]:
Final_Report = []

Final_Report.append(Evaluate_model(model, train_generator, validation_generator, test_generator))

# **Save the Model:**

In [ ]:
model.save('/kaggle/working/cnn_model2.h5')
